**Вариант 10**

**Практическая работа 5. Оптимизация и очистка данных**

Цель работы: научиться уменьшать потребление памяти и ускорять обработку данных в pandas, выявлять и исправлять проблемы качества данных (дубликаты, пропуски, аномальные значения), а также собрать итоговый набор признаков по пользователям.

Задание варианта 10:

1. собрать финальный `DataFrame` `user_features` со всеми агрегированными и нормализованными признаками;
2. проверить его на наличие пропусков и аномальных значений;
3. сохранить результат в формате Parquet и CSV, сравнить размер файлов.

Исходные данные

Используются файлы из папки `for_lab_3` — те же наборы, что и в практических работах 3 и 4:

| Файл | Формат | Содержимое |
|---|---|---|
| `orders_2024_01.csv`, `orders_2024_02.csv`, `orders_2024_03.csv` | CSV | заказы интернет-магазина по месяцам |
| `events_records.json` | JSON (records) | события/клики пользователей |
| `events_lines.json` | JSON Lines | события/клики пользователей |

Структура заказов: `order_id`, `user_id`, `order_datetime`, `product_id`, `price`, `quantity`, `status`.

Структура событий: `event_id`, `user_id`, `event_time`, `event_type`, `page`.

В заказах 15 000 позиций за январь–март 2024 года, в событиях — 4 000 записей. Результаты работы сохраняются в папку `for_lab_5`.

In [1]:
import glob
import json
import os

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

# папка с исходными файлами практической работы
DATA_DIR = 'for_lab_3'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = 'data'

# папка для итоговых файлов
OUT_DIR = 'for_lab_5'
os.makedirs(OUT_DIR, exist_ok=True)

print('Рабочая папка с данными:', os.path.abspath(DATA_DIR))
print('Папка для результатов:  ', os.path.abspath(OUT_DIR))
print()
for path in sorted(glob.glob(os.path.join(DATA_DIR, '*'))):
    print(f'{os.path.basename(path):<25}{os.path.getsize(path):>10} байт')

Рабочая папка с данными: D:\учёба\vsuet\ims\for_lab_3
Папка для результатов:   D:\учёба\vsuet\ims\for_lab_5

events_lines.json            239555 байт
events_records.json          224211 байт
orders_2024_01.csv           244027 байт
orders_2024_02.csv           245248 байт
orders_2024_03.csv           250278 байт
products.xlsx                  6526 байт
user_profile.parquet          11919 байт


**1. Собрать финальный DataFrame user_features со всеми агрегированными и нормализованными признаками**

In [2]:
orders_files = sorted(glob.glob(os.path.join(DATA_DIR, 'orders_2024_*.csv')))

df_orders_raw = pd.concat(
    [pd.read_csv(path) for path in orders_files],
    ignore_index=True,
)

print('Заказы «как есть»:', df_orders_raw.shape)
df_orders_raw.info(memory_usage='deep')

memory_orders_raw = df_orders_raw.memory_usage(deep=True).sum() / 1024**2
print(f'Память исходных заказов: {memory_orders_raw:.2f} МБ')

Заказы «как есть»: (15000, 7)
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        15000 non-null  int64  
 1   user_id         15000 non-null  int64  
 2   order_datetime  15000 non-null  str    
 3   product_id      15000 non-null  int64  
 4   price           15000 non-null  float64
 5   quantity        15000 non-null  int64  
 6   status          15000 non-null  str    
dtypes: float64(1), int64(4), str(2)
memory usage: 1.1 MB
Память исходных заказов: 1.14 МБ


In [3]:
chunk_size = 5_000
total_rows = 0
total_revenue = 0.0
n_chunks = 0

for path in orders_files:
    for chunk in pd.read_csv(path, chunksize=chunk_size):
        total_revenue += (chunk['price'] * chunk['quantity']).sum()
        total_rows += len(chunk)
        n_chunks += 1

full_revenue = (df_orders_raw['price'] * df_orders_raw['quantity']).sum()

print(f'Обработано чанков:      {n_chunks}')
print(f'Строк по чанкам:        {total_rows}')
print(f'Строк в полной таблице: {len(df_orders_raw)}')
print(f'Выручка по чанкам:      {total_revenue:,.2f} руб.')
print(f'Выручка по полной:      {full_revenue:,.2f} руб.')
print(f'Совпадение:             {np.isclose(total_revenue, full_revenue)}')

Обработано чанков:      3
Строк по чанкам:        15000
Строк в полной таблице: 15000
Выручка по чанкам:      384,543,651.79 руб.
Выручка по полной:      384,543,651.79 руб.
Совпадение:             True


In [4]:
orders_dtypes = {
    'order_id': 'int32',
    'user_id': 'int32',
    'product_id': 'int16',
    'price': 'float32',
    'quantity': 'int16',
    'status': 'category',
}

df_orders = pd.concat(
    [
        pd.read_csv(path, dtype=orders_dtypes, parse_dates=['order_datetime'])
        for path in orders_files
    ],
    ignore_index=True,
)

print('Типы оптимизированных заказов:')
print(df_orders.dtypes.to_string())
print()
df_orders.info(memory_usage='deep')

memory_orders_opt = df_orders.memory_usage(deep=True).sum() / 1024**2
print(f'Память оптимизированных заказов: {memory_orders_opt:.2f} МБ')
print(f'Экономия: {100 * (1 - memory_orders_opt / memory_orders_raw):.1f}%')

Типы оптимизированных заказов:
order_id                   int32
user_id                    int32
order_datetime    datetime64[us]
product_id                 int16
price                    float32
quantity                   int16
status                  category

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        15000 non-null  int32         
 1   user_id         15000 non-null  int32         
 2   order_datetime  15000 non-null  datetime64[us]
 3   product_id      15000 non-null  int16         
 4   price           15000 non-null  float32       
 5   quantity        15000 non-null  int16         
 6   status          15000 non-null  category      
dtypes: category(1), datetime64[us](1), float32(1), int16(2), int32(2)
memory usage: 366.5 KB
Память оптимизированных заказов: 0.36 МБ
Экономия: 68.7%


In [5]:
def read_events_records(path):
    """Читает JSON в формате records (список объектов)."""
    with open(path, encoding='utf-8') as f:
        raw = json.load(f)

    df = pd.DataFrame(raw)

    # файл может быть сохранён транспонированным: 5 строк-признаков и 2000 столбцов
    if 'event_id' not in df.columns:
        df = df.T.reset_index(drop=True)
        df.columns = ['event_id', 'user_id', 'event_time', 'event_type', 'page']

    return df


events_records = read_events_records(os.path.join(DATA_DIR, 'events_records.json'))
events_lines = pd.read_json(os.path.join(DATA_DIR, 'events_lines.json'), lines=True)

df_events_raw = pd.concat([events_records, events_lines], ignore_index=True)

print('События «как есть»:', df_events_raw.shape)
df_events_raw.info(memory_usage='deep')

memory_events_raw = df_events_raw.memory_usage(deep=True).sum() / 1024**2
print(f'Память исходных событий: {memory_events_raw:.2f} МБ')

События «как есть»: (4000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   event_id    4000 non-null   object
 1   user_id     4000 non-null   object
 2   event_time  4000 non-null   object
 3   event_type  4000 non-null   object
 4   page        4000 non-null   object
dtypes: object(5)
memory usage: 1.1 MB
Память исходных событий: 1.08 МБ


In [6]:
df_events = df_events_raw.copy()

for col in ['event_id', 'user_id']:
    df_events[col] = pd.to_numeric(df_events[col], errors='coerce').astype('int32')

df_events['event_time'] = pd.to_datetime(df_events['event_time'])
df_events['event_type'] = df_events['event_type'].astype('category')
df_events['page'] = df_events['page'].astype('category')

print('Типы оптимизированных событий:')
print(df_events.dtypes.to_string())
print()
df_events.info(memory_usage='deep')

memory_events_opt = df_events.memory_usage(deep=True).sum() / 1024**2
print(f'Память оптимизированных событий: {memory_events_opt:.2f} МБ')
print(f'Экономия: {100 * (1 - memory_events_opt / memory_events_raw):.1f}%')

Типы оптимизированных событий:
event_id               int32
user_id                int32
event_time    datetime64[us]
event_type          category
page                category

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   event_id    4000 non-null   int32         
 1   user_id     4000 non-null   int32         
 2   event_time  4000 non-null   datetime64[us]
 3   event_type  4000 non-null   category      
 4   page        4000 non-null   category      
dtypes: category(2), datetime64[us](1), int32(2)
memory usage: 71.1 KB
Память оптимизированных событий: 0.07 МБ
Экономия: 93.6%


In [7]:
memory_table = pd.DataFrame(
    [
        {
            'Набор данных': 'Заказы',
            'Строк': len(df_orders),
            'Память до, МБ': memory_orders_raw,
            'Память после, МБ': memory_orders_opt,
            'Экономия, %': 100 * (1 - memory_orders_opt / memory_orders_raw),
        },
        {
            'Набор данных': 'События',
            'Строк': len(df_events),
            'Память до, МБ': memory_events_raw,
            'Память после, МБ': memory_events_opt,
            'Экономия, %': 100 * (1 - memory_events_opt / memory_events_raw),
        },
    ]
)

print('Сравнение потребления памяти:')
print(
    memory_table.to_string(
        index=False,
        formatters={
            'Память до, МБ': '{:.2f}'.format,
            'Память после, МБ': '{:.2f}'.format,
            'Экономия, %': '{:.1f}'.format,
        },
    )
)

Сравнение потребления памяти:
Набор данных  Строк Память до, МБ Память после, МБ Экономия, %
      Заказы  15000          1.14             0.36        68.7
     События   4000          1.08             0.07        93.6


In [8]:
order_key = ['order_id', 'user_id', 'product_id', 'order_datetime']
event_key = ['event_id', 'user_id', 'event_time', 'event_type', 'page']

dup_orders_full = df_orders.duplicated()
dup_orders_key = df_orders.duplicated(subset=order_key, keep='first')

dup_events_full = df_events.duplicated()
dup_events_key = df_events.duplicated(subset=event_key, keep='first')

print('Дубликаты в заказах:')
print(f'  полные:                 {dup_orders_full.sum()}')
print(f'  по бизнес-ключу:        {dup_orders_key.sum()}')

print()
print('Дубликаты в событиях:')
print(f'  полные:                 {dup_events_full.sum()}')
print(f'  по бизнес-ключу:        {dup_events_key.sum()}')

df_orders_nodup = df_orders[~dup_orders_key].copy()
df_events_nodup = df_events[~dup_events_key].copy()

dup_table = pd.DataFrame(
    [
        {
            'Набор данных': 'Заказы',
            'Строк до': len(df_orders),
            'Дубликатов': int(dup_orders_key.sum()),
            'Строк после': len(df_orders_nodup),
            'Уменьшение, %': 100 * (1 - len(df_orders_nodup) / len(df_orders)),
        },
        {
            'Набор данных': 'События',
            'Строк до': len(df_events),
            'Дубликатов': int(dup_events_key.sum()),
            'Строк после': len(df_events_nodup),
            'Уменьшение, %': 100 * (1 - len(df_events_nodup) / len(df_events)),
        },
    ]
)

print()
print('Размер таблиц до и после удаления дубликатов:')
print(
    dup_table.to_string(
        index=False,
        formatters={'Уменьшение, %': '{:.2f}'.format},
    )
)

Дубликаты в заказах:
  полные:                 0
  по бизнес-ключу:        0

Дубликаты в событиях:
  полные:                 0
  по бизнес-ключу:        0

Размер таблиц до и после удаления дубликатов:
Набор данных  Строк до  Дубликатов  Строк после Уменьшение, %
      Заказы     15000           0        15000          0.00
     События      4000           0         4000          0.00


In [9]:
bad_values = ['', ' ', 'none', 'null', 'unknown', '-']


def count_bad_values(series):
    """Считает неявные пропуски в столбце."""
    return int(series.astype('string').str.strip().str.lower().isin(bad_values).sum())


bad_summary = pd.DataFrame(
    {
        'Столбец': ['status', 'event_type', 'page'],
        'Неявных пропусков': [
            count_bad_values(df_orders_nodup['status']),
            count_bad_values(df_events_nodup['event_type']),
            count_bad_values(df_events_nodup['page']),
        ],
    }
)

print('Неявные пропуски до очистки:')
print(bad_summary.to_string(index=False))

# очистка статусов заказов
df_orders_nodup['status_clean'] = (
    df_orders_nodup['status']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace(bad_values, pd.NA)
    .astype('category')
)

# очистка типов событий и страниц
df_events_nodup['event_type_clean'] = (
    df_events_nodup['event_type']
    .astype('string')
    .str.strip()
    .replace(bad_values, pd.NA)
    .astype('category')
)

df_events_nodup['page_clean'] = (
    df_events_nodup['page']
    .astype('string')
    .str.strip()
    .replace(bad_values, pd.NA)
    .astype('category')
)

print()
print('Статусы после очистки:')
print(df_orders_nodup['status_clean'].value_counts(dropna=False).to_string())
print()
print('Типы событий после очистки:')
print(df_events_nodup['event_type_clean'].value_counts(dropna=False).to_string())
print()
print('Пропуски после очистки:')
print(f'  в заказах: {int(df_orders_nodup.isna().sum().sum())}')
print(f'  в событиях: {int(df_events_nodup.isna().sum().sum())}')

Неявные пропуски до очистки:
   Столбец  Неявных пропусков
    status                  0
event_type                  0
      page                  0

Статусы после очистки:
status_clean
paid         12032
cancelled     2212
refunded       756

Типы событий после очистки:


event_type_clean
click          819
purchase       807
page_view      804
scroll         788
add_to_cart    782

Пропуски после очистки:
  в заказах: 0
  в событиях: 0


In [10]:
print('Описательная статистика числовых признаков заказов:')
print(df_orders_nodup[['price', 'quantity']].describe().round(2).to_string())

print()
print('Цены <= 0:        ', int((df_orders_nodup['price'] <= 0).sum()))
print('Количество <= 0:  ', int((df_orders_nodup['quantity'] <= 0).sum()))
print('Максимальное количество в заказе:', int(df_orders_nodup['quantity'].max()))

# страховка: неположительные цены превращаем в NaN (в наших данных таких нет)
df_orders_nodup['price_clean'] = df_orders_nodup['price'].where(df_orders_nodup['price'] > 0)
df_orders_nodup['revenue'] = df_orders_nodup['price_clean'] * df_orders_nodup['quantity']

print()
print('Цен, заменённых на NaN:', int(df_orders_nodup['price_clean'].isna().sum()))
print(df_orders_nodup[['price', 'price_clean', 'quantity', 'revenue']].head().to_string())

Описательная статистика числовых признаков заказов:
          price  quantity
count  15000.00  15000.00
mean   10207.38      2.50
std     5657.48      1.12
min      300.26      1.00
25%     5304.96      2.00
50%    10267.21      3.00
75%    15050.53      4.00
max    19998.96      4.00

Цены <= 0:         0
Количество <= 0:   0
Максимальное количество в заказе: 4

Цен, заменённых на NaN: 0
          price   price_clean  quantity       revenue
0  14038.150391  14038.150391         1  14038.150391
1   9172.599609   9172.599609         1   9172.599609
2   9035.259766   9035.259766         2  18070.519531
3  15234.330078  15234.330078         3  45702.992188
4   4134.379883   4134.379883         2   8268.759766


In [11]:
mask_product_page = df_events_nodup['page_clean'].str.startswith('/product/', na=False)
df_events_products = df_events_nodup[mask_product_page].copy()

df_events_products['product_id_from_page'] = (
    df_events_products['page_clean']
    .str.extract(r'/product/(\d+)', expand=False)
    .astype('float')
    .astype('Int32')
)

product_ids_orders = set(df_orders_nodup['product_id'].unique())
product_ids_pages = set(df_events_products['product_id_from_page'].dropna().unique().tolist())

print(f'Событий на страницах товаров: {len(df_events_products)} '
      f'({100 * len(df_events_products) / len(df_events_nodup):.1f}% от всех)')
print(f'Корректно извлечён product_id: {int(df_events_products["product_id_from_page"].notna().sum())}')
print(f'Уникальных product_id в заказах: {len(product_ids_orders)}')
print(f'Уникальных product_id в URL:    {len(product_ids_pages)}')
print(f'Совпадающих product_id:         {len(product_ids_orders & product_ids_pages)}')
print()
print(df_events_products[['page_clean', 'product_id_from_page']].head().to_string(index=False))

Событий на страницах товаров: 2832 (70.8% от всех)
Корректно извлечён product_id: 2832
Уникальных product_id в заказах: 30
Уникальных product_id в URL:    30
Совпадающих product_id:         30

 page_clean  product_id_from_page
/product/26                    26
/product/28                    28
/product/27                    27
 /product/1                     1
/product/19                    19


In [12]:
status_map = {
    'paid': 'paid',
    'cancelled': 'cancelled',
    'refunded': 'refunded',
}

df_orders_nodup['status_group'] = (
    df_orders_nodup['status_clean']
    .astype('string')
    .map(lambda x: status_map.get(x, 'other'))
    .astype('category')
)

print('Группы статусов:')
print(df_orders_nodup['status_group'].value_counts().to_string())

Группы статусов:
status_group
paid         12032
cancelled     2212
refunded       756


In [13]:
orders_by_user = (
    df_orders_nodup
    .groupby('user_id', observed=True)
    .agg(
        orders_count=('order_id', 'nunique'),
        items_count=('quantity', 'sum'),
        revenue_sum=('revenue', 'sum'),
        cancelled_count=('status_group', lambda s: (s == 'cancelled').sum()),
        first_order=('order_datetime', 'min'),
        last_order=('order_datetime', 'max'),
    )
)

orders_by_user['avg_check'] = orders_by_user['revenue_sum'] / orders_by_user['orders_count']
orders_by_user['cancelled_share'] = orders_by_user['cancelled_count'] / orders_by_user['orders_count']

events_by_user = (
    df_events_nodup
    .groupby('user_id', observed=True)
    .agg(
        events_count=('event_id', 'count'),
        page_view=('event_type_clean', lambda s: (s == 'page_view').sum()),
        click=('event_type_clean', lambda s: (s == 'click').sum()),
        scroll=('event_type_clean', lambda s: (s == 'scroll').sum()),
        add_to_cart=('event_type_clean', lambda s: (s == 'add_to_cart').sum()),
        purchase=('event_type_clean', lambda s: (s == 'purchase').sum()),
        first_event=('event_time', 'min'),
        last_event=('event_time', 'max'),
    )
)

print('Агрегаты по заказам:', orders_by_user.shape)
print(orders_by_user.head().to_string())
print()
print('Агрегаты по событиям:', events_by_user.shape)
print(events_by_user.head().to_string())

Агрегаты по заказам: (100, 8)
         orders_count  items_count  revenue_sum  cancelled_count         first_order          last_order     avg_check  cancelled_share
user_id                                                                                                                                
1001              177          454   4435767.00               26 2024-01-02 02:51:37 2024-03-31 21:32:19  25060.830508         0.146893
1002              123          281   2830004.50               16 2024-01-02 12:57:34 2024-03-31 19:45:16  23008.166667         0.130081
1003              156          389   4025266.25               19 2024-01-02 02:02:44 2024-03-31 19:36:26  25802.988782         0.121795
1004              147          380   3788030.50               17 2024-01-01 00:19:34 2024-03-31 20:01:40  25768.914966         0.115646
1005              141          346   3185653.50               23 2024-01-01 01:11:51 2024-03-31 18:16:10  22593.287234         0.163121

Агрегаты по событ

In [14]:
count_cols = [
    'orders_count', 'items_count', 'revenue_sum', 'cancelled_count', 'cancelled_share',
    'avg_check', 'events_count', 'page_view', 'click', 'scroll', 'add_to_cart', 'purchase',
]

user_features = orders_by_user.join(events_by_user, how='outer')

# у пользователей без заказов/событий счётчики должны быть нулями
user_features[count_cols] = user_features[count_cols].fillna(0)

# общий период активности по двум источникам
user_features['first_activity'] = user_features[['first_order', 'first_event']].min(axis=1)
user_features['last_activity'] = user_features[['last_order', 'last_event']].max(axis=1)
user_features['lifetime_days'] = (
    user_features['last_activity'] - user_features['first_activity']
).dt.days

# плотность событий на заказ (защита от деления на ноль)
user_features['events_per_order'] = (
    user_features['events_count'] / user_features['orders_count'].replace(0, np.nan)
)

user_features['has_orders'] = (user_features['orders_count'] > 0).astype('int8')
user_features['has_events'] = (user_features['events_count'] > 0).astype('int8')

print('user_features:', user_features.shape)
print(user_features[
    ['orders_count', 'revenue_sum', 'avg_check', 'events_count', 'purchase', 'lifetime_days']
].head(10).to_string())

user_features: (100, 22)
         orders_count  revenue_sum     avg_check  events_count  purchase  lifetime_days
user_id                                                                                
1001              177   4435767.00  25060.830508            54        16             90
1002              123   2830004.50  23008.166667            41        12             89
1003              156   4025266.25  25802.988782            39        16             89
1004              147   3788030.50  25768.914966            37        15             90
1005              141   3185653.50  22593.287234            44        13             90
1006              161   4160641.00  25842.490683            36         5             89
1007              157   4064170.75  25886.437898            29         7             90
1008              122   2886592.50  23660.594262            43         8             90
1009              147   3545781.75  24120.964286            42         8             90
1010   

In [15]:
norm_cols = ['orders_count', 'items_count', 'revenue_sum', 'avg_check', 'events_count']

for col in norm_cols:
    user_features[f'{col}_minmax'] = (
        (user_features[col] - user_features[col].min())
        / (user_features[col].max() - user_features[col].min())
    )
    user_features[f'{col}_z'] = (
        (user_features[col] - user_features[col].mean())
        / user_features[col].std(ddof=0)
    )

minmax_cols = [f'{col}_minmax' for col in norm_cols]
z_cols = [f'{col}_z' for col in norm_cols]

print('Мин–макс нормализация:')
print(user_features[minmax_cols].describe().round(3).to_string())
print()
print('Z-нормализация:')
print(user_features[z_cols].describe().round(3).to_string())

Мин–макс нормализация:
       orders_count_minmax  items_count_minmax  revenue_sum_minmax  avg_check_minmax  events_count_minmax
count              100.000             100.000             100.000           100.000              100.000
mean                 0.509               0.547               0.524             0.472                0.400
std                  0.233               0.202               0.217             0.239                0.184
min                  0.000               0.000               0.000             0.000                0.000
25%                  0.345               0.408               0.377             0.303                0.269
50%                  0.555               0.592               0.561             0.494                0.375
75%                  0.673               0.678               0.671             0.668                0.475
max                  1.000               1.000               1.000             1.000                1.000

Z-нормализация:


       orders_count_z  items_count_z  revenue_sum_z  avg_check_z  events_count_z
count         100.000        100.000        100.000      100.000         100.000
mean            0.000          0.000         -0.000       -0.000          -0.000
std             1.005          1.005          1.005        1.005           1.005
min            -2.195         -2.720         -2.421       -1.982          -2.183
25%            -0.706         -0.694         -0.680       -0.711          -0.716
50%             0.196          0.226          0.172        0.093          -0.136
75%             0.706          0.650          0.679        0.823           0.409
max             2.117          2.252          2.202        2.216           3.274


**2. Проверить его на наличие пропусков и аномальных значений**

In [16]:
missing = user_features.isna().sum()
print('Пропуски по столбцам (только ненулевые):')
print(missing[missing > 0].to_string() if (missing > 0).any() else '  пропусков нет')
print()
print('Всего пропусков:', int(missing.sum()))
print('Пользователей в заказах: ', orders_by_user.shape[0])
print('Пользователей в событиях:', events_by_user.shape[0])
print('Пользователей в итоговой таблице:', user_features.shape[0])
print()
print('Пользователей только с заказами: ', int(((user_features['orders_count'] > 0) & (user_features['events_count'] == 0)).sum()))
print('Пользователей только с событиями:', int(((user_features['orders_count'] == 0) & (user_features['events_count'] > 0)).sum()))
print('Пользователей с заказами и событиями:', int(((user_features['orders_count'] > 0) & (user_features['events_count'] > 0)).sum()))

Пропуски по столбцам (только ненулевые):


  пропусков нет

Всего пропусков: 0
Пользователей в заказах:  100
Пользователей в событиях: 100
Пользователей в итоговой таблице: 100

Пользователей только с заказами:  0
Пользователей только с событиями: 0
Пользователей с заказами и событиями: 100


In [17]:
key_metrics = ['orders_count', 'items_count', 'revenue_sum', 'avg_check', 'events_count']

print('Основные метрики user_features:')
print(user_features[key_metrics].describe().round(2).to_string())
print()


def iqr_outliers(series):
    """Возвращает маску выбросов по правилу 1.5 * IQR."""
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    return (series < low) | (series > high), low, high


outlier_rows = []
for col in key_metrics:
    mask, low, high = iqr_outliers(user_features[col])
    outlier_rows.append(
        {
            'Признак': col,
            'Нижняя граница': low,
            'Верхняя граница': high,
            'Выбросов': int(mask.sum()),
            'Максимум': user_features[col].max(),
        }
    )

print('Выбросы по правилу 1.5 * IQR:')
print(
    pd.DataFrame(outlier_rows).to_string(
        index=False,
        formatters={
            'Нижняя граница': '{:,.2f}'.format,
            'Верхняя граница': '{:,.2f}'.format,
            'Максимум': '{:,.2f}'.format,
        },
    )
)

print()
print('Пользователей с |z| > 3 по выручке:',
      int((user_features['revenue_sum_z'].abs() > 3).sum()))
print('Отрицательных значений в счётчиках:',
      int((user_features[['orders_count', 'items_count', 'revenue_sum', 'events_count']] < 0).sum().sum()))

print()
print('Топ-5 пользователей по выручке:')
print(
    user_features
    .nlargest(5, 'revenue_sum')
    [['orders_count', 'revenue_sum', 'avg_check', 'events_count']]
    .round(2)
    .to_string()
)

print()
print('Топ-5 пользователей по числу событий:')
print(
    user_features
    .nlargest(5, 'events_count')
    [['orders_count', 'revenue_sum', 'events_count', 'purchase']]
    .to_string()
)

Основные метрики user_features:
       orders_count  items_count  revenue_sum  avg_check  events_count
count        100.00       100.00       100.00     100.00        100.00
mean         150.00       375.65   3845436.75   25621.61         40.00
std           12.82        34.97    421576.28    1568.59          7.37
min          122.00       281.00   2830004.50   22528.76         24.00
25%          141.00       351.50   3560102.44   24511.20         34.75
50%          152.50       383.50   3917571.88   25767.36         39.00
75%          159.00       398.25   4130283.38   26905.62         43.00
max          177.00       454.00   4769059.00   29079.63         64.00

Выбросы по правилу 1.5 * IQR:
     Признак Нижняя граница Верхняя граница  Выбросов     Максимум
orders_count         114.00          186.00         0       177.00
 items_count         281.38          468.38         1       454.00
 revenue_sum   2,704,831.03    4,985,554.78         0 4,769,059.00
   avg_check      20,919.57   

         orders_count  revenue_sum  avg_check  events_count
user_id                                                    
1066              164    4769059.0   29079.63            43
1081              169    4715929.5   27904.91            34
1012              171    4613593.5   26980.08            39
1017              163    4558873.0   27968.55            34
1027              171    4557348.0   26651.16            36

Топ-5 пользователей по числу событий:
         orders_count  revenue_sum  events_count  purchase
user_id                                                   
1091              147   4204414.50            64        19
1051              153   4117942.25            59         6
1072              150   3958219.25            57        11
1093              133   3260214.50            55        14
1001              177   4435767.00            54        16


**3. Сохранить результат в формате Parquet и CSV, сравнить размер файлов**

In [18]:
features_out = user_features.reset_index()

parquet_path = os.path.join(OUT_DIR, 'user_features.parquet')
csv_path = os.path.join(OUT_DIR, 'user_features.csv')

features_out.to_parquet(parquet_path, index=False)
features_out.to_csv(csv_path, index=False)

size_parquet = os.path.getsize(parquet_path)
size_csv = os.path.getsize(csv_path)

size_table = pd.DataFrame(
    [
        {'Формат': 'Parquet', 'Файл': parquet_path, 'Размер, байт': size_parquet, 'Размер, КБ': size_parquet / 1024},
        {'Формат': 'CSV', 'Файл': csv_path, 'Размер, байт': size_csv, 'Размер, КБ': size_csv / 1024},
    ]
)

print('Сохранённые файлы:')
print(
    size_table.to_string(
        index=False,
        formatters={'Размер, байт': '{:,}'.format, 'Размер, КБ': '{:.1f}'.format},
    )
)
print()
print(f'CSV больше Parquet в {size_csv / size_parquet:.2f} раза '
      f'({100 * (size_csv / size_parquet - 1):+.1f}%)')
print(f'Память итоговой таблицы: {features_out.memory_usage(deep=True).sum() / 1024**2:.3f} МБ')
print(f'Размер таблицы: {features_out.shape[0]} строк × {features_out.shape[1]} столбцов')

Сохранённые файлы:
 Формат                            Файл Размер, байт Размер, КБ
Parquet for_lab_5\user_features.parquet       35,654       34.8
    CSV     for_lab_5\user_features.csv       39,031       38.1

CSV больше Parquet в 1.09 раза (+9.5%)
Память итоговой таблицы: 0.022 МБ
Размер таблицы: 100 строк × 33 столбцов


In [19]:
parquet_back = pd.read_parquet(parquet_path)
csv_back = pd.read_csv(csv_path)

print('Проверка чтения обратно:')
print(f'  Parquet: {parquet_back.shape}')
print(f'  CSV:     {csv_back.shape}')
print(f'  Размеры совпадают: {parquet_back.shape == csv_back.shape}')
print()
print('Типы данных после чтения Parquet (фрагмент):')
print(parquet_back[['user_id', 'orders_count', 'revenue_sum', 'avg_check', 'first_order']].dtypes.to_string())
print()
print('Типы данных после чтения CSV (фрагмент):')
print(csv_back[['user_id', 'orders_count', 'revenue_sum', 'avg_check', 'first_order']].dtypes.to_string())
print()
print('Первые 5 строк сохранённого набора:')
print(parquet_back[['user_id', 'orders_count', 'revenue_sum', 'events_count', 'lifetime_days']].head().to_string(index=False))

Проверка чтения обратно:


  Parquet: (100, 33)
  CSV:     (100, 33)
  Размеры совпадают: True

Типы данных после чтения Parquet (фрагмент):
user_id                  int32
orders_count             int64
revenue_sum            float32
avg_check              float64
first_order     datetime64[us]

Типы данных после чтения CSV (фрагмент):
user_id           int64
orders_count      int64
revenue_sum     float64
avg_check       float64
first_order         str

Первые 5 строк сохранённого набора:
 user_id  orders_count  revenue_sum  events_count  lifetime_days
    1001           177   4435767.00            54             90
    1002           123   2830004.50            41             89
    1003           156   4025266.25            39             89
    1004           147   3788030.50            37             90
    1005           141   3185653.50            44             90
